# Project 1 — Exploratory Analysis of Breast Cancer Molecular Subtypes (TCGA-BRCA)

**Author:** [Your name]
**Date:** 2026
**Contact:** [email / LinkedIn / portfolio website]

---

## Executive summary

> **To be filled in last — 4 to 6 lines maximum.**
> This summary should be readable in 30 seconds by a non-technical prospect.
>
> Suggested structure:
> - **Context** — why this question matters clinically
> - **Approach** — what methods, on what data
> - **Key finding** — 1 or 2 striking numbers
> - **Implication** — why it matters for medical decision-making

---

## Table of contents

1. [Biological context & research question](#1-context)
2. [Data loading and cleaning](#2-cleaning)
3. [Descriptive exploratory analysis](#3-eda)
4. [Statistical testing](#4-stats)
5. [Synthesis visualizations](#5-viz)
6. [Discussion, limitations and next steps](#6-discussion)


<a id="1-context"></a>
## 1. Biological context & research question

### 1.1 Why breast cancer?

Breast cancer is the most common cancer in women worldwide. It is also a **molecularly heterogeneous** disease: under similar histological appearances, tumors fall into several **intrinsic molecular subtypes** (PAM50 classification):

- **Luminal A** — hormone receptor positive (ER+/PR+), HER2-, low proliferation. Good prognosis.
- **Luminal B** — hormone receptor positive, higher proliferation. Intermediate prognosis.
- **HER2-enriched** — HER2 overexpression. Aggressive but targetable with anti-HER2 therapies.
- **Basal-like** (~triple-negative) — ER-, PR-, HER2-. Aggressive, limited therapeutic options.
- **Normal-like** — profile close to healthy breast tissue.

This classification has **direct clinical implications**: treatment choice, follow-up intensity, 5- and 10-year prognosis.

### 1.2 Research question

> **Which clinical characteristics distinguish the molecular subtypes of breast cancer in the TCGA-BRCA cohort, and are these differences statistically significant after correction for multiple testing?**

### 1.3 Operational sub-questions

- Do subtypes differ in **age at diagnosis**?
- Do subtypes differ in **tumor stage** at diagnosis?
- Do subtypes differ in **vital status** at last follow-up?
- Are **missing data** distributed at random, or do they reveal collection biases?

### 1.4 Stated methodological choices

- **Data source** — TCGA-BRCA via the GDC Data Portal (portal.gdc.cancer.gov)
- **Type of analysis** — exploratory, descriptive and inferential (no predictive modeling at this stage)
- **Statistical tests** — selected based on variable types and normality (justified in section 4)
- **Multiple testing correction** — Benjamini-Hochberg (FDR), α=0.05 threshold


<a id="2-cleaning"></a>
## 2. Data loading and cleaning

### 2.1 Retrieving the data

**Recommended method for this project:** download clinical files directly via the GDC Data Portal.

**Step-by-step procedure:**

1. Go to https://portal.gdc.cancer.gov/
2. Click "Projects" → filter "Program: TCGA" → select "TCGA-BRCA"
3. Click "Save New Cohort" → name it (e.g., `brca_project1`)
4. Go to "Cohort Builder" → check filters (here none, we take the whole cohort)
5. Click "Repository" → "Files" tab → filter:
   - Data Category: **Clinical**
   - Data Format: **bcr xml** or **bcr biotab** (TSV format is easier to start with)
6. Click "Add All Files to Cart" → "Cart" → "Download → Clinical (TSV)"

**Faster alternative to get started:** use the `cbio-py` or `pycbio` Python package to query the cBioPortal API.

**For molecular data** (PAM50 subtype, gene expression) — add as a complement from cBioPortal (study `brca_tcga_pan_can_atlas_2018`).

### 2.2 Imports and configuration


In [ ]:
# Imports — standard scientific data analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Statistics
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Global configuration
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Consistent plotting style (to reuse across the whole portfolio)
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (8, 5)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_DIR = Path('./data/raw')
PROCESSED_DIR = Path('./data/processed')
FIGURES_DIR = Path('./figures')

for d in [DATA_DIR, PROCESSED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment ready.")


### 2.3 Loading the raw clinical file

The GDC provides several clinical files for BRCA. The main ones:
- `clinical.tsv` — demographic and diagnostic data (one row per patient)
- `exposure.tsv` — exposures (tobacco, alcohol)
- `family_history.tsv`
- `follow_up.tsv` — **multiple rows per patient** (follow-up may be repeated over time)

**First pitfall:** in `follow_up.tsv`, the same patient may appear several times. We need to keep the most recent record.


In [ ]:
# Load the main clinical file
# Separator is a tab, and missing values are encoded in several ways
clinical_raw = pd.read_csv(
    DATA_DIR / 'clinical.tsv',
    sep='\t',
    na_values=["'--", '[Not Available]', '[Not Applicable]', '[Unknown]', '[Not Evaluated]', 'unknown']
)

print(f"Shape: {clinical_raw.shape}")
print(f"Unique patients: {clinical_raw['case_submitter_id'].nunique()}")
clinical_raw.head()


### 2.4 Column inventory and completeness rates

Before any cleaning, we need to know what we have. A good reflex: produce a table with, for each column, its **type**, **number of unique values**, and **percentage of missing values**.


In [ ]:
def column_inventory(df: pd.DataFrame) -> pd.DataFrame:
    """Return an inventory of the columns of a DataFrame.

    For each column: dtype, number of unique values, % of NaN.
    Useful for quickly identifying which columns to keep / clean / drop.
    """
    inventory = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(dropna=True),
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().sum() / len(df) * 100).round(1),
    })
    return inventory.sort_values('pct_missing', ascending=False)

inventory = column_inventory(clinical_raw)
print(f"Total columns: {len(inventory)}")
print(f"Columns with >50% missing: {(inventory['pct_missing'] > 50).sum()}")
inventory.head(20)


### 2.5 Selecting variables of interest

> **Modeling decision to document:** keep only variables relevant to the research question. This avoids the temptation of "data dredging" (running tests everywhere in the hope of finding something).

Variables retained for this analysis:

| Variable | Type | Justification |
|---|---|---|
| `case_submitter_id` | identifier | join key for molecular data |
| `age_at_diagnosis` | numeric (days) | between-subtype comparison |
| `gender` | categorical | control (male breast cancer ≈1%) |
| `race`, `ethnicity` | categorical | detection of cohort bias |
| `vital_status` | categorical | "Alive" / "Dead" at last follow-up |
| `ajcc_pathologic_stage` | ordinal | tumor stage at diagnosis |
| `primary_diagnosis` | categorical | histological subtype |


In [ ]:
variables_of_interest = [
    'case_submitter_id',
    'age_at_diagnosis',
    'gender',
    'race',
    'ethnicity',
    'vital_status',
    'ajcc_pathologic_stage',
    'primary_diagnosis',
]

# Safety check: are all columns present?
missing_cols = [c for c in variables_of_interest if c not in clinical_raw.columns]
if missing_cols:
    print(f"⚠️  Missing columns: {missing_cols}")
else:
    print("✓ All variables of interest are present")

clinical = clinical_raw[variables_of_interest].copy()
clinical.head()


### 2.6 Handling duplicates

In the GDC `clinical.tsv` file, the same patient can appear several times if multiple specimens or follow-ups have been recorded.

**Deduplication rule:** keep one row per `case_submitter_id`, preferring the most complete row (the fewest NaN).


In [ ]:
print(f"Before deduplication: {len(clinical)} rows, {clinical['case_submitter_id'].nunique()} patients")

# Count of non-NaN values per row, used as a 'completeness' score
clinical['_completeness'] = clinical.notna().sum(axis=1)

# Sort by completeness (descending), then keep the first row per patient
clinical = (
    clinical.sort_values('_completeness', ascending=False)
            .drop_duplicates(subset='case_submitter_id', keep='first')
            .drop(columns='_completeness')
            .reset_index(drop=True)
)

print(f"After deduplication: {len(clinical)} rows, {clinical['case_submitter_id'].nunique()} patients")


### 2.7 Age conversion

`age_at_diagnosis` is encoded in **days** in TCGA. We convert to years for readability.


In [ ]:
clinical['age_years'] = clinical['age_at_diagnosis'] / 365.25

# Sanity check: realistic distribution?
print(clinical['age_years'].describe().round(1))


### 2.8 Cleaning tumor stage

The `ajcc_pathologic_stage` field contains substages (IA, IB, IIA, etc.). For analysis, we collapse them into main stages (I, II, III, IV).


In [ ]:
def simplify_stage(stage):
    """Collapse AJCC substages into main stages."""
    if pd.isna(stage):
        return np.nan
    stage = str(stage).replace('Stage ', '').strip()
    # Stage X = not determinable
    if stage in ('X', 'x', ''):
        return np.nan
    # Keep only the main roman numerals
    for s in ['IV', 'III', 'II', 'I']:
        if stage.startswith(s):
            return s
    return np.nan

clinical['stage_simplified'] = clinical['ajcc_pathologic_stage'].apply(simplify_stage)

# Logical stage order for visualizations
stage_order = ['I', 'II', 'III', 'IV']
clinical['stage_simplified'] = pd.Categorical(
    clinical['stage_simplified'], categories=stage_order, ordered=True
)

clinical['stage_simplified'].value_counts(dropna=False)


### 2.9 Merging with molecular subtypes (PAM50)

> **To adapt:** this step assumes you have separately downloaded a file containing the PAM50 classification (from cBioPortal, study `brca_tcga_pan_can_atlas_2018`, column `Subtype` or `SUBTYPE`).

If you don't have this file yet, you can either download it now, or continue the analysis without subtypes for now and add it later.


In [ ]:
# Load molecular subtypes
# Adapt the path and column name to your source file
try:
    subtypes = pd.read_csv(DATA_DIR / 'pam50_subtypes.tsv', sep='\t')
    # Harmonize the identifier: cBioPortal barcodes sometimes carry a -01 suffix
    subtypes['case_submitter_id'] = subtypes['SAMPLE_ID'].str[:12]
    subtypes = subtypes[['case_submitter_id', 'SUBTYPE']].drop_duplicates('case_submitter_id')

    # Join
    clinical = clinical.merge(subtypes, on='case_submitter_id', how='left')

    # Renaming for readability
    clinical = clinical.rename(columns={'SUBTYPE': 'pam50_subtype'})
    print(f"✓ Subtypes merged. Coverage: {clinical['pam50_subtype'].notna().sum()}/{len(clinical)}")
    print(clinical['pam50_subtype'].value_counts(dropna=False))
except FileNotFoundError:
    print("⚠️  PAM50 file not found — analysis will continue without this variable.")
    print("    Download `data_clinical_sample.txt` from cBioPortal (brca_tcga_pan_can_atlas_2018).")


### 2.10 Saving the cleaned dataset


In [ ]:
# Save to avoid re-running the full cleaning pipeline every time
clinical.to_csv(PROCESSED_DIR / 'brca_clinical_clean.csv', index=False)
print(f"✓ Cleaned data saved ({len(clinical)} patients, {clinical.shape[1]} variables)")


<a id="3-eda"></a>
## 3. Descriptive exploratory analysis

> **Guiding principle:** before any statistical test, look at the data. Each visualization should answer a specific question and be commented on.

### 3.1 Cohort overview


In [ ]:
print(f"=== TCGA-BRCA COHORT AFTER CLEANING ===")
print(f"Number of patients: {len(clinical)}")
print(f"Female / Male: {(clinical['gender'] == 'female').sum()} / {(clinical['gender'] == 'male').sum()}")
print(f"Median age at diagnosis: {clinical['age_years'].median():.1f} years")
print(f"Range: {clinical['age_years'].min():.0f} - {clinical['age_years'].max():.0f} years")
print(f"Observed deaths: {(clinical['vital_status'] == 'Dead').sum()} ({(clinical['vital_status'] == 'Dead').mean()*100:.1f}%)")


### 3.2 Distribution of age at diagnosis

**Question:** is the age distribution approximately normal? (This will guide the choice of tests in section 4.)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram + density
sns.histplot(clinical['age_years'].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of age at diagnosis")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Number of patients")
axes[0].axvline(clinical['age_years'].median(), color='red', linestyle='--', label=f"Median: {clinical['age_years'].median():.1f}")
axes[0].legend()

# Q-Q plot to assess normality
stats.probplot(clinical['age_years'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title("Q-Q plot vs normal distribution")

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_age_distribution.png', bbox_inches='tight')
plt.show()

# Formal normality test (Shapiro-Wilk)
stat, p = stats.shapiro(clinical['age_years'].dropna().sample(min(5000, clinical['age_years'].notna().sum()), random_state=42))
print(f"\nShapiro-Wilk: statistic={stat:.4f}, p={p:.2e}")
print("→ p < 0.05: reject normality assumption." if p < 0.05
      else "→ p ≥ 0.05: compatible with a normal distribution.")


**Interpretation:**
> *To complete after running.* Typically on TCGA-BRCA, the age distribution is slightly skewed with a tail toward younger ages. We will therefore use **non-parametric tests** by default (Mann-Whitney, Kruskal-Wallis) rather than t-tests.

### 3.3 Distribution by molecular subtype


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))

    subtype_order = ['BRCA_LumA', 'BRCA_LumB', 'BRCA_Her2', 'BRCA_Basal', 'BRCA_Normal']
    available_subtypes = [s for s in subtype_order if s in clinical['pam50_subtype'].unique()]

    counts = clinical['pam50_subtype'].value_counts().reindex(available_subtypes)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='colorblind')

    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(int(v)), ha='center', fontweight='bold')

    ax.set_title("Patient counts by PAM50 molecular subtype")
    ax.set_xlabel("Subtype")
    ax.set_ylabel("Number of patients")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_subtype_counts.png', bbox_inches='tight')
    plt.show()
else:
    print("Subtypes not available — section skipped.")


### 3.4 Age by subtype — first visual impression


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years',
        order=available_subtypes,
        palette='colorblind', ax=ax
    )
    sns.stripplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years',
        order=available_subtypes,
        color='black', alpha=0.2, size=2, ax=ax
    )
    ax.set_title("Age at diagnosis by PAM50 subtype")
    ax.set_xlabel("Subtype")
    ax.set_ylabel("Age (years)")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_age_by_subtype.png', bbox_inches='tight')
    plt.show()


### 3.5 Tumor stage by subtype


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    crosstab = pd.crosstab(
        clinical['pam50_subtype'],
        clinical['stage_simplified'],
        normalize='index'
    ) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    crosstab.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')
    ax.set_title("Tumor stage distribution by subtype (% within subtype)")
    ax.set_xlabel("Subtype")
    ax.set_ylabel("% of patients")
    ax.legend(title='Stage', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_stage_by_subtype.png', bbox_inches='tight')
    plt.show()


<a id="4-stats"></a>
## 4. Statistical testing

> **Guiding principle:** for each test, we document (1) the question, (2) the test choice and its justification, (3) the underlying assumptions verified, (4) the interpretation.

### 4.1 Does age differ significantly between subtypes?

**Question:** `age_years` (continuous) vs `pam50_subtype` (categorical, >2 groups).

**Test selection:**
- Continuous, non-normal variable (see section 3.2) → **non-parametric test**
- More than 2 groups → **Kruskal-Wallis** (non-parametric equivalent of ANOVA)
- If significant → **Dunn** post-hoc with Bonferroni correction


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    # Build the groups
    groups = [
        clinical.loc[clinical['pam50_subtype'] == s, 'age_years'].dropna().values
        for s in available_subtypes
    ]

    # Global test
    stat_kw, p_kw = stats.kruskal(*groups)
    print(f"Kruskal-Wallis: H={stat_kw:.3f}, p={p_kw:.2e}")
    print(f"→ {'Significant difference detected' if p_kw < 0.05 else 'No significant difference'}")


### 4.2 Is tumor stage associated with subtype?

**Question:** `stage_simplified` (ordinal categorical) vs `pam50_subtype` (categorical).

**Test selection:**
- Two categorical variables → **Chi² test of independence**
- Check that all expected counts ≥ 5 (otherwise → Fisher's exact test)


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    contingency = pd.crosstab(clinical['pam50_subtype'], clinical['stage_simplified'])
    print("Contingency table (observed counts):")
    print(contingency)

    chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

    # Check the application condition
    min_expected = expected.min()
    print(f"\nχ² = {chi2:.3f}, df = {dof}, p = {p_chi2:.2e}")
    print(f"Minimum expected count: {min_expected:.1f}")

    if min_expected < 5:
        print("⚠️  At least one expected count < 5: use Fisher's exact test (via R or statsmodels)")


### 4.3 Does mortality differ between subtypes?

**Question:** `vital_status` (binary) vs `pam50_subtype` (categorical).

**Test selection:** Chi² test of independence.

> **Important note for what's next:** this analysis does not account for follow-up time. A proper survival analysis (Kaplan-Meier, log-rank, Cox) will be the focus of **Project 4**. Here we use a simplistic proxy and state it explicitly.


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    contingency_vital = pd.crosstab(clinical['pam50_subtype'], clinical['vital_status'])
    print(contingency_vital)

    chi2, p, dof, _ = stats.chi2_contingency(contingency_vital)
    print(f"\nχ² = {chi2:.3f}, df = {dof}, p = {p:.2e}")


### 4.4 Correction for multiple testing

We ran 3 tests on the same cohort. The probability of at least one false positive at α=0.05 is: 1 − 0.95³ ≈ 14%. So we apply a correction.

**Method chosen:** Benjamini-Hochberg (FDR). More powerful than Bonferroni, suited to settings where we expect several true positives.


In [ ]:
# To adapt with the actual p-values obtained above
results = pd.DataFrame({
    'test': ['Age ~ subtype', 'Stage ~ subtype', 'Mortality ~ subtype'],
    'p_value': [p_kw, p_chi2, p],  # variable names as defined above
})

rejected, p_corrected, _, _ = multipletests(results['p_value'], alpha=0.05, method='fdr_bh')
results['p_corrected_fdr'] = p_corrected
results['significant_fdr'] = rejected

print(results.to_string(index=False))


**Interpretation:**
> *To complete after running.* For each test, state whether the conclusion changes after FDR correction, and formulate the conclusion in clinical language (not just statistical).

### 4.5 A note on effect vs significance

> **Always mention this in a client-facing report:** a significant p-value says **nothing** about the **magnitude** of the effect. With n ≈ 1000 patients, a 2-year age gap between subtypes can be highly statistically significant yet clinically negligible.
>
> Always complement a p-value with an **effect size** (median difference, Cramér's V, η²) and a **confidence interval**.


In [ ]:
# Example: median age difference between LumA and Basal, with bootstrap CI
if 'pam50_subtype' in clinical.columns and {'BRCA_LumA', 'BRCA_Basal'}.issubset(set(clinical['pam50_subtype'].dropna())):
    luma = clinical.loc[clinical['pam50_subtype'] == 'BRCA_LumA', 'age_years'].dropna().values
    basal = clinical.loc[clinical['pam50_subtype'] == 'BRCA_Basal', 'age_years'].dropna().values

    obs_diff = np.median(luma) - np.median(basal)

    # Bootstrap (10,000 replications)
    n_boot = 10_000
    boot_diffs = np.empty(n_boot)
    rng = np.random.default_rng(RANDOM_STATE)
    for i in range(n_boot):
        b_luma = rng.choice(luma, size=len(luma), replace=True)
        b_basal = rng.choice(basal, size=len(basal), replace=True)
        boot_diffs[i] = np.median(b_luma) - np.median(b_basal)

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    print(f"Median age difference (LumA − Basal): {obs_diff:+.1f} years")
    print(f"95% bootstrap CI: [{ci_low:+.1f}, {ci_high:+.1f}] years")


<a id="5-viz"></a>
## 5. Synthesis visualizations

> **Guiding principle:** these figures are intended for a **clinician** or a **biotech decision-maker**, not a statistician. They should be readable in 5 seconds and tell the essential story.

### 5.1 "Hero" figure — summary of inter-subtype differences

A single figure combining age, stage and mortality by subtype. To refine for your portfolio.


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel A: age
    sns.boxplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years', order=available_subtypes,
        palette='colorblind', ax=axes[0]
    )
    axes[0].set_title("A. Age at diagnosis")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Age (years)")
    axes[0].tick_params(axis='x', rotation=30)

    # Panel B: stage (% stage III-IV)
    advanced_stage = (
        clinical.dropna(subset=['pam50_subtype', 'stage_simplified'])
        .assign(advanced=lambda d: d['stage_simplified'].isin(['III', 'IV']))
        .groupby('pam50_subtype')['advanced'].mean() * 100
    ).reindex(available_subtypes)

    sns.barplot(x=advanced_stage.index, y=advanced_stage.values, palette='colorblind', ax=axes[1])
    axes[1].set_title("B. % diagnoses at stage III-IV")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("% patients")
    axes[1].tick_params(axis='x', rotation=30)

    # Panel C: observed mortality
    mortality = (
        clinical.dropna(subset=['pam50_subtype'])
        .assign(dead=lambda d: d['vital_status'] == 'Dead')
        .groupby('pam50_subtype')['dead'].mean() * 100
    ).reindex(available_subtypes)

    sns.barplot(x=mortality.index, y=mortality.values, palette='colorblind', ax=axes[2])
    axes[2].set_title("C. % deaths at last follow-up")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("% patients")
    axes[2].tick_params(axis='x', rotation=30)

    fig.suptitle("Clinical characteristics by PAM50 subtype (TCGA-BRCA)", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'hero_subtype_summary.png', bbox_inches='tight', dpi=300)
    plt.show()


### 5.2 Data completeness heatmap

A figure often forgotten but very appreciated by clients: where are the gaps in the dataset? It's a proof of scientific rigor.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
missing_matrix = clinical[variables_of_interest[1:]].isna().T  # exclude the ID
sns.heatmap(missing_matrix, cbar=False, cmap='RdYlGn_r', ax=ax)
ax.set_title("Missing data map (red = missing)")
ax.set_xlabel("Patients (ordered)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_missingness_heatmap.png', bbox_inches='tight')
plt.show()


<a id="6-discussion"></a>
## 6. Discussion, limitations and next steps

### 6.1 Main findings

> *To be written after full execution, in 3-5 bullet points, in clinical language.*
>
> Example structure:
> - The Basal-like subtype is diagnosed at a significantly younger age (median X years vs Y years for Luminal A, p_FDR = ...)
> - The Basal-like subtype is over-represented in advanced stages (Z% vs W% for Luminal A)
> - ...

### 6.2 Methodological limitations

To identify and **always mention for a client**:

1. **TCGA selection bias** — the TCGA cohort is not representative of the general population (US academic centers, strict inclusion criteria). Conclusions don't extrapolate directly to other populations.

2. **Non-random missing data** — some fields (HER2 status, treatments) have >30% missing rates. Patients with complete data may differ from others.

3. **Mortality ≠ survival** — comparing `vital_status` without accounting for follow-up time is a crude proxy. **Project 4** will provide a rigorous survival analysis (Kaplan-Meier, Cox).

4. **No external validation** — all analyses rely on a single dataset. Ideally we would confirm on an independent cohort (METABRIC, for example).

5. **Multiple testing** — only 3 tests here, but in practice one often runs dozens. FDR correction is applied but remains an approximation.

### 6.3 Next steps (Projects 2-4)

- **Project 2** — differential gene expression analysis between Basal-like and Luminal A subtypes → identify differentially expressed genes and assess consistency with the literature.
- **Project 3** — interactive Streamlit dashboard allowing a clinician to filter the cohort and explore these analyses without coding.
- **Project 4** — survival analysis (Kaplan-Meier by subtype, Cox model adjusted for age and stage) to actually quantify differential prognosis.

### 6.4 Reproducibility

- **Code** — available on GitHub at [URL]
- **Data** — TCGA-BRCA, accessible via portal.gdc.cancer.gov (free account)
- **Environment** — `requirements.txt` provided in the repo
- **Seed** — `RANDOM_STATE = 42` for all stochastic operations

---

## References

1. The Cancer Genome Atlas Network. *Comprehensive molecular portraits of human breast tumours.* Nature 490, 61–70 (2012).
2. Parker JS et al. *Supervised risk predictor of breast cancer based on intrinsic subtypes (PAM50).* JCO 27 (2009).
3. Benjamini Y, Hochberg Y. *Controlling the false discovery rate.* J R Stat Soc B 57 (1995).
